# Instalações

In [ ]:
%pip install -r requirements.txt

In [ ]:
print("Dependências centralizadas em requirements.txt e instaladas na célula anterior.")

# Pré Processamento

In [64]:
from pathlib import Path

from causal_discovery import CausalPreprocessor, load_daily_delhi_climate, summarize_ensemble
from causal_discovery.methods import (
    run_classical_granger,
    run_dynotears,
    run_fci,
    run_lpcmci,
    run_neural_granger,
    run_pcmci,
    run_score_based_ges,
    run_var_lingam,
    )

DATA_PATH = Path("DailyDelhiClimateTrain.csv")
SELECTED_COLUMNS = ["meantemp", "humidity", "wind_speed", "meanpressure"]

raw_data = load_daily_delhi_climate(DATA_PATH)
data = raw_data[SELECTED_COLUMNS].copy()

preprocessor = CausalPreprocessor(
    data,
    significance_level=0.05,
    decomposition_period=30,
    )
processed_data = preprocessor.fit_transform(
    make_stationary=True,
    normalize=True,
    remove_trend=False,
    max_diffs=2,
    )

max_lag = 5
preprocessing_summary = preprocessor.summary()

print("--- Dados Originais ---")
print(data.head())
print("\n--- Resumo do Pré-processamento ---")
print(preprocessing_summary)
print("\n--- Dados Processados ---")
print(processed_data.head())

--- Dados Originais ---
             meantemp   humidity  wind_speed  meanpressure
date                                                      
2013-01-01  10.000000  84.500000    0.000000   1015.666667
2013-01-02   7.400000  92.000000    2.980000   1017.800000
2013-01-03   7.166667  87.000000    4.633333   1018.666667
2013-01-04   8.666667  71.333333    1.233333   1017.166667
2013-01-05   6.000000  86.833333    3.700000   1016.500000

--- Resumo do Pré-processamento ---
         column  differencing_order
0      meantemp                   1
1      humidity                   0
2    wind_speed                   0
3  meanpressure                   0

--- Dados Processados ---
            meantemp  humidity  wind_speed  meanpressure
date                                                    
2013-01-02 -1.556049  1.864438   -0.839570      0.037166
2013-01-03 -0.139645  1.566076   -0.476847      0.041975
2013-01-04  0.897720  0.631208   -1.222768      0.033652
2013-01-05 -1.595947  1.556131   -

# PCMCI

In [ ]:
pcmci_results = run_pcmci(
    processed_data,
    max_lag=max_lag,
    pc_alpha=0.05,
    alpha_level=0.05,
    )

lpcmci_results = run_lpcmci(
    processed_data,
    max_lag=max_lag,
    pc_alpha=0.05,
    )

print("--- Resultados PCMCI ---")
print(pcmci_results.head())
print(f"Total de links PCMCI: {len(pcmci_results)}")

print("\n--- Resultados LPCMCI ---")
print(lpcmci_results.head())
print(f"Total de links LPCMCI: {len(lpcmci_results)}")

# Granger clássico e Neural Granger cMLP

In [ ]:
classical_granger_results = run_classical_granger(
    processed_data,
    max_lag=max_lag,
    alpha=0.05,
    )

neural_granger_results = run_neural_granger(
    processed_data,
    max_lag=max_lag,
    hidden_layer_sizes=(32,),
    lambda_group=0.5,
    lambda_ridge=0.01,
    learning_rate=0.005,
    max_iter=800,
    check_every=100,
    patience=5,
    )

print("--- Resultados Classical Granger ---")
print(classical_granger_results.head())
print(f"Total de links Classical Granger: {len(classical_granger_results)}")

print("\n--- Resultados Neural Granger cMLP ---")
print(neural_granger_results.head())
print(f"Total de links Neural Granger: {len(neural_granger_results)}")


# VAR-LiNGAM, DYNOTEARS, GES e FCI

In [ ]:
var_lingam_results = run_var_lingam(
    processed_data,
    max_lag=max_lag,
    )

dynotears_results = run_dynotears(
    processed_data,
    max_lag=max_lag,
    lambda1=0.01,
    max_iter=50,
    w_threshold=0.05,
    standardize=True,
    )

ges_results = run_score_based_ges(
    processed_data,
    max_lag=max_lag,
    score_func="local_score_BIC",
    max_parents=6,
    penalty_discount=0.5,
    )

fci_results = run_fci(
    processed_data,
    max_lag=max_lag,
    alpha=0.05,
    independence_test_method="fisherz",
    include_partial=False,
    )

print("--- Resultados VAR-LiNGAM ---")
print(var_lingam_results.head())
print(f"Total de links VAR-LiNGAM: {len(var_lingam_results)}")

print("\n--- Resultados DYNOTEARS ---")
print(dynotears_results.head())
print(f"Total de links DYNOTEARS: {len(dynotears_results)}")

print("\n--- Resultados GES ---")
print(ges_results.head())
print(f"Total de links GES: {len(ges_results)}")

print("\n--- Resultados FCI ---")
print(fci_results.head())
print(f"Total de links FCI definitivamente orientados: {len(fci_results)}")


# Ensemble

In [ ]:
all_results = [
    pcmci_results,
    lpcmci_results,
    classical_granger_results,
    neural_granger_results,
    var_lingam_results,
    dynotears_results,
    ges_results,
    fci_results,
    ]

method_sizes = {
    "PCMCI": len(pcmci_results),
    "LPCMCI": len(lpcmci_results),
    "ClassicalGranger": len(classical_granger_results),
    "NeuralGrangercMLP": len(neural_granger_results),
    "VARLiNGAM": len(var_lingam_results),
    "DYNOTEARS": len(dynotears_results),
    "GES": len(ges_results),
    "FCI": len(fci_results),
    }

ensemble_summary = summarize_ensemble(all_results, min_votes=2)

print("--- Quantidade de links por método ---")
print(method_sizes)
print("\n--- Sumário do Ensemble (mínimo de 2 votos) ---")
print(ensemble_summary.head(20))


# Ensemble probabilístico e seleção robusta

In [ ]:
from causal_discovery import compute_method_consistency

results_by_method = {
    "PCMCI": pcmci_results,
    "LPCMCI": lpcmci_results,
    "ClassicalGranger": classical_granger_results,
    "NeuralGrangercMLP": neural_granger_results,
    "VARLiNGAM": var_lingam_results,
    "DYNOTEARS": dynotears_results,
    "GES": ges_results,
    "FCI": fci_results,
    }

consistency_matrix = compute_method_consistency(results_by_method)

print("--- Consistência entre métodos (Jaccard) ---")
print(consistency_matrix.round(3))


In [ ]:
from causal_discovery import summarize_probabilistic_ensemble

ensemble_prob_summary = summarize_probabilistic_ensemble(
    all_results,
    min_votes=2,
    prior_edge_probability=0.1,
    posterior_weight=0.7,
    confidence_level=0.95,
    method_names=list(method_sizes.keys()),
    )

print("--- Sumário probabilístico do ensemble ---")
print(
    ensemble_prob_summary[
        [
            "source",
            "target",
            "lag",
            "votes",
            "edge_probability",
            "uncertainty",
            "confidence",
            ]
        ].head(20)
    )


In [ ]:
import importlib
import math
import os
import pandas as pd

from causal_discovery.methods import (
    run_dynotears,
    run_lpcmci,
    run_pcmci,
    run_score_based_ges,
    run_var_lingam,
    )

if "processed_data" not in globals() or "max_lag" not in globals():
    raise RuntimeError("Execute as células 1 a 3 antes desta etapa para gerar processed_data e max_lag.")

import causal_discovery.ensemble as ensemble_mod
import causal_discovery.expert_knowledge as expert_mod
import causal_discovery.ensemble_selection as ensemble_selection_mod

importlib.reload(ensemble_mod)
importlib.reload(expert_mod)
importlib.reload(ensemble_selection_mod)
select_robust_ensemble_combination = ensemble_selection_mod.select_robust_ensemble_combination

# O painel pode alternar entre os modos sem precisar reconstruir esta configuração.
quick_mode = globals().get("quick_mode", False)

all_candidate_methods = {
    "PCMCI": run_pcmci,
    "LPCMCI": run_lpcmci,
    "VARLiNGAM": run_var_lingam,
    "DYNOTEARS": run_dynotears,
    "GES": run_score_based_ges,
    }

all_candidate_method_kwargs = {
    "PCMCI": {"max_lag": max_lag, "pc_alpha": 0.05, "alpha_level": 0.05},
    "LPCMCI": {"max_lag": max_lag, "pc_alpha": 0.05},
    "VARLiNGAM": {"max_lag": max_lag},
    "DYNOTEARS": {"max_lag": max_lag, "lambda1": 0.01, "max_iter": 50, "w_threshold": 0.05, "standardize": True},
    "GES": {"max_lag": max_lag, "score_func": "local_score_BIC", "max_parents": 6, "penalty_discount": 0.5},
    }

all_method_weights = {
    "PCMCI": 1.0,
    "LPCMCI": 1.10,
    "VARLiNGAM": 0.95,
    "DYNOTEARS": 0.90,
    "GES": 1.0,
    }


def select_runtime_methods(quick_mode):
    names = ["PCMCI", "LPCMCI", "GES"] if quick_mode else list(all_candidate_methods)
    methods = {name: all_candidate_methods[name] for name in names}
    kwargs = {name: all_candidate_method_kwargs[name] for name in names}
    weights = {name: all_method_weights[name] for name in names}
    return names, methods, kwargs, weights


search_method_names, candidate_methods, candidate_method_kwargs, method_weights = select_runtime_methods(quick_mode)

# Regras iniciais do especialista. A interface da próxima célula pode editar esta lista.
expert_knowledge_default = [
    {
        "source": "humidity",
        "target": "meantemp",
        "lag": 0,
        "relation": "strong",
        "confidence": 0.90,
        "constraint": "soft",
        "prior_probability": 0.90,
    },
    {
        "source": "meantemp",
        "target": "meanpressure",
        "lag": 2,
        "relation": "weak",
        "confidence": 0.70,
        "constraint": "soft",
        "prior_probability": 0.45,
    },
    {
        "source": "meanpressure",
        "target": "humidity",
        "lag": 0,
        "relation": "none",
        "confidence": 0.95,
        "constraint": "hard",
    },
    ]
expert_knowledge = globals().get("expert_knowledge", expert_knowledge_default)


def build_selection_args(quick_mode=None, n_bootstrap=None, parallel_jobs=None):
    quick_mode = globals().get("quick_mode", False) if quick_mode is None else bool(quick_mode)
    n_bootstrap = (4 if quick_mode else 8) if n_bootstrap is None else int(n_bootstrap)
    parallel_jobs = max(1, min(4, (os.cpu_count() or 2) - 1)) if parallel_jobs is None else int(parallel_jobs)
    return {
        "min_methods": 2,
        "max_methods": 2 if quick_mode else 3,
        "min_votes": 2,
        "n_bootstrap": n_bootstrap,
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.6,
        "selection_probability_threshold": 0.55,
        "prior_edge_probability": 0.1,
        "posterior_weight": 0.7,
        "confidence_level": 0.95,
        "random_state": 42,
        "precompute_runs": True,
        "parallel_jobs": parallel_jobs,
        "max_bootstrap_seconds": 240 if quick_mode else 900,
    }


def print_selection_cost(selection_args, method_names):
    num_methods = len(method_names)
    min_methods = selection_args["min_methods"]
    max_methods = min(selection_args["max_methods"], num_methods)
    n_bootstrap = selection_args["n_bootstrap"]
    num_combinations = sum(math.comb(num_methods, k) for k in range(min_methods, max_methods + 1))
    naive_method_runs = sum(math.comb(num_methods, k) * k for k in range(min_methods, max_methods + 1)) * (n_bootstrap + 1)
    optimized_method_runs = num_methods * (n_bootstrap + 1)

    print("--- Estimativa de custo computacional ---")
    print(f"Métodos avaliados: {method_names}")
    print(f"Combinações: {num_combinations}")
    print(f"Execuções de método (aprox. sem otimização): {naive_method_runs}")
    print(f"Execuções de método (aprox. com precompute_runs): {optimized_method_runs}")
    print(f"Paralelismo (jobs): {selection_args['parallel_jobs']}")
    print(f"Limite de tempo do bootstrap (s): {selection_args['max_bootstrap_seconds']}")


common_selection_args = build_selection_args(quick_mode=quick_mode)
print_selection_cost(common_selection_args, search_method_names)
print(f"\nRegras especialistas carregadas: {len(expert_knowledge)}")
print("Use a próxima célula para revisar as regras e os parâmetros antes da execução.")

# Interface interativa e visualização

In [ ]:
import importlib
import pandas as pd

import causal_discovery.visualization as visualization_mod
importlib.reload(visualization_mod)

create_advanced_expert_dashboard = visualization_mod.create_advanced_expert_dashboard


def pipeline_runner(
    quick_mode, n_bootstrap, parallel_jobs, expert_knowledge,
    processed_data, candidate_methods, candidate_method_kwargs, method_weights
):
    from causal_discovery.ensemble_selection import select_robust_ensemble_combination

    selected_names = ["PCMCI", "LPCMCI", "GES"] if quick_mode else list(candidate_methods)
    selected_methods = {name: candidate_methods[name] for name in selected_names}
    selected_kwargs = {name: candidate_method_kwargs[name] for name in selected_names}
    selected_weights = {name: method_weights[name] for name in selected_names}
    selected_expert_knowledge = list(expert_knowledge)
    selection_args = build_selection_args(
        quick_mode=quick_mode,
        n_bootstrap=n_bootstrap,
        parallel_jobs=parallel_jobs,
    )

    selection_result_expert = select_robust_ensemble_combination(
        processed_data,
        selected_methods,
        method_kwargs=selected_kwargs,
        method_weights=selected_weights,
        expert_knowledge=selected_expert_knowledge,
        **selection_args,
    )

    best_eval = selection_result_expert["best_evaluation"]
    expert_summary = best_eval["probabilistic_summary"]
    consistency = best_eval["consistency"]

    globals()["quick_mode"] = bool(quick_mode)
    globals()["search_method_names"] = selected_names
    globals()["candidate_methods"] = selected_methods
    globals()["candidate_method_kwargs"] = selected_kwargs
    globals()["method_weights"] = selected_weights
    globals()["expert_knowledge"] = selected_expert_knowledge
    globals()["last_selection_args"] = selection_args
    globals()["selection_result_expert"] = selection_result_expert
    globals()["selection_result"] = selection_result_expert
    globals()["ranking"] = selection_result_expert["ranking"]
    globals()["best_eval"] = best_eval

    return {
        "probabilistic_summary": expert_summary,
        "consistency": consistency,
        "selection_result": selection_result_expert,
        "expert_knowledge": selected_expert_knowledge,
        "selection_args": selection_args,
        "method_names": selected_names,
    }


all_column_nodes = list(processed_data.columns)

# O painel recebe todos os métodos; Quick/Full escolhe o subconjunto no momento da execução.
dashboard_ui = create_advanced_expert_dashboard(
    processed_data=processed_data,
    candidate_methods=all_candidate_methods,
    candidate_method_kwargs=all_candidate_method_kwargs,
    method_weights=all_method_weights,
    all_nodes=all_column_nodes,
    pipeline_callback=pipeline_runner,
    initial_expert_knowledge=expert_knowledge,
    initial_quick_mode=quick_mode,
    initial_n_bootstrap=common_selection_args["n_bootstrap"],
    initial_parallel_jobs=common_selection_args["parallel_jobs"],
)

# Execução robusta com as regras atuais do especialista

In [ ]:
from causal_discovery.ensemble_selection import select_robust_ensemble_combination

if "dashboard_ui" in globals():
    expert_knowledge = list(dashboard_ui.current_rules)
    quick_mode = bool(dashboard_ui.quick_mode_control.value)
    current_bootstraps = int(dashboard_ui.bootstrap_control.value)
    current_parallel_jobs = int(dashboard_ui.parallel_jobs_control.value)
else:
    current_bootstraps = 4 if quick_mode else 8
    current_parallel_jobs = max(1, min(4, (os.cpu_count() or 2) - 1))

search_method_names, candidate_methods, candidate_method_kwargs, method_weights = select_runtime_methods(quick_mode)
selection_args = build_selection_args(
    quick_mode=quick_mode,
    n_bootstrap=current_bootstraps,
    parallel_jobs=current_parallel_jobs,
)
common_selection_args = selection_args

dashboard_result = getattr(dashboard_ui, "pipeline_result", None) if "dashboard_ui" in globals() else None
dashboard_rules = dashboard_result.get("expert_knowledge") if isinstance(dashboard_result, dict) else None
dashboard_args = dashboard_result.get("selection_args") if isinstance(dashboard_result, dict) else None
dashboard_methods = dashboard_result.get("method_names") if isinstance(dashboard_result, dict) else None
dashboard_selection = dashboard_result.get("selection_result") if isinstance(dashboard_result, dict) else None

if (
    dashboard_selection is not None
    and dashboard_rules == expert_knowledge
    and dashboard_args == selection_args
    and dashboard_methods == search_method_names
):
    print("--- Usando resultado com especialista já gerado pela interface ---")
    selection_result_expert = dashboard_selection
else:
    print("--- Executando com as regras e os parâmetros atuais ---")
    selection_result_expert = select_robust_ensemble_combination(
        processed_data,
        candidate_methods,
        method_kwargs=candidate_method_kwargs,
        method_weights=method_weights,
        expert_knowledge=expert_knowledge,
        **selection_args,
    )

data_only_signature = (
    id(processed_data),
    tuple(search_method_names),
    tuple(sorted(selection_args.items())),
)
data_only_cache = globals().setdefault("_selection_data_only_cache", {})
if data_only_signature not in data_only_cache:
    data_only_cache[data_only_signature] = select_robust_ensemble_combination(
        processed_data,
        candidate_methods,
        method_kwargs=candidate_method_kwargs,
        **selection_args,
    )
selection_result_data_only = data_only_cache[data_only_signature]

globals()["selection_result_expert"] = selection_result_expert
globals()["selection_result_data_only"] = selection_result_data_only

def describe_expert_rules(rules):
    rows = []
    for index, rule in enumerate(rules, start=1):
        rows.append(
            {
                "rule_id": f"R{index}",
                "source": rule.get("source"),
                "target": rule.get("target"),
                "lag": rule.get("lag"),
                "relation": rule.get("relation"),
                "constraint": rule.get("constraint"),
                "confidence": rule.get("confidence"),
                "prior_probability": rule.get("prior_probability"),
            }
        )
    return pd.DataFrame(rows)


def attach_matching_rule_ids(summary, rules):
    if summary is None or summary.empty or not rules:
        output = summary.copy() if summary is not None else pd.DataFrame()
        output["matched_rules"] = ""
        return output

    output = summary.copy()
    matched = []
    for _, edge in output.iterrows():
        rule_ids = []
        for index, rule in enumerate(rules, start=1):
            same_source = str(edge.get("source")) == str(rule.get("source"))
            same_target = str(edge.get("target")) == str(rule.get("target"))
            rule_lag = rule.get("lag")
            if pd.isna(rule_lag):
                same_lag = True
            else:
                try:
                    same_lag = int(edge.get("lag")) == int(rule_lag)
                except Exception:
                    same_lag = False
            if same_source and same_target and same_lag:
                rule_ids.append(f"R{index}")
        matched.append(", ".join(rule_ids))
    output["matched_rules"] = matched
    return output


def summarize_rule_impacts(expert_edges):
    if expert_edges.empty:
        return pd.DataFrame(
            columns=["rule_id", "affected_edges", "total_votes", "max_edge_probability", "methods"]
        )

    rows = []
    for rule_id in sorted({item.strip() for text in expert_edges["matched_rules"] for item in text.split(",") if item.strip()}):
        subset = expert_edges[expert_edges["matched_rules"].str.contains(rule_id, regex=False)]
        method_names = sorted({method for methods in subset.get("method", []) for method in (methods if isinstance(methods, list) else [methods])})
        rows.append(
            {
                "rule_id": rule_id,
                "affected_edges": len(subset),
                "total_votes": int(subset["votes"].sum()) if "votes" in subset else 0,
                "max_edge_probability": float(subset["edge_probability"].max()) if "edge_probability" in subset else float("nan"),
                "methods": method_names,
            }
        )
    return pd.DataFrame(rows)


ranking_data_only = selection_result_data_only["ranking"].copy()
ranking_data_only["scenario"] = "dados_apenas"
ranking_data_only["expert_rules"] = "nenhuma"

active_rule_table = describe_expert_rules(expert_knowledge)
active_rule_ids = ", ".join(active_rule_table["rule_id"].tolist()) if not active_rule_table.empty else "nenhuma"

ranking_expert = selection_result_expert["ranking"].copy()
ranking_expert["scenario"] = "dados_especialista"
ranking_expert["expert_rules"] = active_rule_ids

ranking_compare = pd.concat([ranking_data_only.head(5), ranking_expert.head(5)], ignore_index=True)

print("\n--- Regras especialistas ativas nesta execução ---")
print(active_rule_table if not active_rule_table.empty else "Nenhuma regra especialista ativa.")

print("\n--- Comparação dos rankings (dados vs especialista) ---")
print(
    ranking_compare[
        [
            "scenario",
            "combination",
            "expert_rules",
            "performance_score",
            "mean_stability",
            "mean_edge_probability",
            "mean_confidence",
        ]
    ]
)

print("\n--- Melhor combinação sem especialista ---")
print(selection_result_data_only["best_combination"])

print("\n--- Melhor combinação com especialista ---")
print(selection_result_expert["best_combination"])

# Seguimos com cenário especialista para visualização, benchmark auxiliar e análises posteriores.
selection_result = selection_result_expert
ranking = selection_result["ranking"]
best_eval = selection_result["best_evaluation"]

print("\n--- Métricas da melhor combinação (com especialista) ---")
print(best_eval["metrics"])

print("\n--- Arestas estáveis da melhor combinação (com especialista) ---")
stable_edges = best_eval["stability"]
stable_edges = stable_edges[stable_edges["stability_selected"]]
print(stable_edges.head(20))

expert_edges = attach_matching_rule_ids(best_eval["probabilistic_summary"], expert_knowledge)
expert_edges = expert_edges[expert_edges["matched_rules"].astype(str) != ""]
expert_columns = [
    "matched_rules",
    "source",
    "target",
    "lag",
    "votes",
    "method",
    "edge_probability",
    "expert_adjustment",
    "expert_effect",
    "expert_confidence",
]
expert_columns = [column for column in expert_columns if column in expert_edges.columns]

print("\n--- Resumo de impacto por regra especialista ---")
rule_impact_summary = summarize_rule_impacts(expert_edges)
print(rule_impact_summary if not rule_impact_summary.empty else "Nenhuma regra correspondeu diretamente às arestas da melhor combinação.")

print("\n--- Arestas afetadas por regras especialistas ---")
if expert_edges.empty:
    print("Nenhuma aresta da melhor combinação correspondeu diretamente às regras especialistas ativas.")
else:
    print(expert_edges[expert_columns].head(30))

# Benchmark e Validação com Ground Truth

In [ ]:
from causal_discovery.benchmark import generate_synthetic_timeseries, compute_structural_metrics

# 1. Gera Dataset Controlado
df_synth, ground_truth = generate_synthetic_timeseries(n_samples=500)

print("--- Ground Truth Esperado ---")
print(ground_truth)

# 2. Executa Ensemble no sintético
from causal_discovery.ensemble_selection import select_robust_ensemble_combination

benchmark_methods = {
    "PCMCI": all_candidate_methods["PCMCI"],
    "LPCMCI": all_candidate_methods["LPCMCI"],
}
benchmark_method_kwargs = {
    "PCMCI": {**all_candidate_method_kwargs["PCMCI"], "max_lag": 2},
    "LPCMCI": {**all_candidate_method_kwargs["LPCMCI"], "max_lag": 2},
}

synthetic_selection = select_robust_ensemble_combination(
    df_synth,
    benchmark_methods, # usando apenas PCMCI e LPCMCI
    method_kwargs=benchmark_method_kwargs,
    min_methods=1,
    max_methods=2,
    min_votes=1,
    n_bootstrap=2, # ultra rápido
    selection_probability_threshold=0.5
)

synth_summary = synthetic_selection["best_evaluation"]["probabilistic_summary"]

# 3. Calcula as Métricas SHD, Precision, F1-Score
metrics = compute_structural_metrics(synth_summary, ground_truth, prob_threshold=0.5)

print("\n--- Resultados do Ensemble (Probabilístico) ---")
print(synth_summary[["source", "target", "lag", "edge_probability"]])

print("\n--- Métricas Estruturais (Benchmark) ---")
for k, v in metrics.items():
    print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
from causal_discovery.benchmark import inject_noise_regime_change

# 4. Injeção de Ruído e Quebra de Regime (Teste de Robustez)
print("--- Teste de Robustez a Ruído e Quebra de Regime ---")

# Cria uma cópia com quebra de regime na metade (índice 250) e 3x mais ruído
df_noisy = inject_noise_regime_change(df_synth, index_change=250, noise_multiplier=3.0)

# Roda a mesma seleção robusta na série corrompida
noisy_selection = select_robust_ensemble_combination(
    df_noisy,
    benchmark_methods,
    method_kwargs=benchmark_method_kwargs,
    min_methods=1,
    max_methods=2,
    min_votes=1,
    n_bootstrap=2,
    selection_probability_threshold=0.5
)

noisy_summary = noisy_selection["best_evaluation"]["probabilistic_summary"]
noisy_metrics = compute_structural_metrics(noisy_summary, ground_truth, prob_threshold=0.5)

print("Métricas originais (Limpo):", {k: round(v, 2) if isinstance(v, float) else v for k,v in metrics.items() if k in ['f1_score', 'structural_hamming_distance']})
print("Métricas com Ruído Severo:", {k: round(v, 2) if isinstance(v, float) else v for k,v in noisy_metrics.items() if k in ['f1_score', 'structural_hamming_distance']})